In [1]:
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
import joblib

In [3]:
df=pd.read_csv('c:/ml_datasets/Language Detection.csv')
df

,Text,Language
0,"Nature, in the broadest sense, is the natural...",English
1,"""Nature"" can refer to the phenomena of the phy...",English
2,"The study of nature is a large, if not the onl...",English
3,"Although humans are part of nature, human acti...",English
4,[1] The word nature is borrowed from the Old F...,English
...,...,...
10332,ನಿಮ್ಮ ತಪ್ಪು ಏನು ಬಂದಿದೆಯೆಂದರೆ ಆ ದಿನದಿಂದ ನಿಮಗೆ ಒ...,Kannada
10333,ನಾರ್ಸಿಸಾ ತಾನು ಮೊದಲಿಗೆ ಹೆಣಗಾಡುತ್ತಿದ್ದ ಮಾರ್ಗಗಳನ್...,Kannada
10334,ಹೇಗೆ ' ನಾರ್ಸಿಸಿಸಮ್ ಈಗ ಮರಿಯನ್ ಅವರಿಗೆ ಸಂಭವಿಸಿದ ಎ...,Kannada
10335,ಅವಳು ಈಗ ಹೆಚ್ಚು ಚಿನ್ನದ ಬ್ರೆಡ್ ಬಯಸುವುದಿಲ್ಲ ಎಂದು ...,Kannada


In [4]:
df.Language.value_counts()

Language
English       1385
French        1014
Spanish        819
Portugeese     739
Italian        698
Russian        692
Sweedish       676
Malayalam      594
Dutch          546
Arabic         536
Turkish        474
German         470
Tamil          469
Danish         428
Kannada        369
Greek          365
Hindi           63
Name: count, dtype: int64

In [5]:
corpus=df.Text
target=df.Language

In [6]:
tv=TfidfVectorizer()
X=tv.fit_transform(corpus)
print(len(tv.get_feature_names_out()))

model=LogisticRegression(class_weight='balanced')
d=cross_validate(model,X,target,cv=5,return_train_score=True)
print(d['train_score'].mean())
print(d['test_score'].mean())


39928
0.99293797158806
0.9442706564143737


In [7]:
tv=TfidfVectorizer(lowercase=True,stop_words=None)
pipe=Pipeline([('tv',tv),('model',LogisticRegression(class_weight='balanced'))])
pipe.fit(corpus,target)

Pipeline(steps=[('tv', TfidfVectorizer()),
                ('model', LogisticRegression(class_weight='balanced'))])

In [8]:
joblib.dump(pipe,"lang_det.pkl")

['lang_det.pkl']

In [11]:
df=pd.read_csv("c:/ml_datasets/spam_ham.txt",sep="\t")

df

,Target,Msg
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [34]:
df.Target.value_counts()

Target
ham     4825
spam     747
Name: count, dtype: int64

In [12]:
df.Target=df.Target.map({'spam':0,'ham':1})
df

,Target,Msg
0,1,"Go until jurong point, crazy.. Available only ..."
1,1,Ok lar... Joking wif u oni...
2,0,Free entry in 2 a wkly comp to win FA Cup fina...
3,1,U dun say so early hor... U c already then say...
4,1,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,0,This is the 2nd time we have tried 2 contact u...
5568,1,Will ü b going to esplanade fr home?
5569,1,"Pity, * was in mood for that. So...any other s..."
5570,1,The guy did some bitching but I acted like i'd...


In [13]:
sws=list(ENGLISH_STOP_WORDS)
sws.remove("not")
sws.remove("no")
for w in sws:
    if w.endswith("n't"):
        sws.remove(w)

In [14]:
from sklearn.metrics import f1_score

In [15]:
corpus=df.Msg
target=df.Target

tv=TfidfVectorizer(lowercase=True,stop_words=sws)
X=tv.fit_transform(corpus)
print(len(tv.get_feature_names_out()))

model=MultinomialNB()
d=cross_validate(model,X,target,cv=5,return_train_score=True,scoring='f1')
print(d['train_score'].mean())
print(d['test_score'].mean())

8446
0.9889829181162171
0.9823951639996977


In [16]:
model=MultinomialNB()
model.fit(X,target)


MultinomialNB()

In [17]:
X.toarray().shape

(5572, 8446)

In [18]:
target.value_counts()

Target
1    4825
0     747
Name: count, dtype: int64

In [19]:
pred=model.predict(X)
f1_score(target,pred,average=None)

array([0.93361884, 0.99045469])

In [20]:
from imblearn.over_sampling import SMOTE

In [21]:
corpus=df.Msg
target=df.Target

tv=TfidfVectorizer(lowercase=True,stop_words=sws)
X=tv.fit_transform(corpus)

smote=SMOTE()
X_new,y=smote.fit_resample(X,target)
X_new.shape

(9650, 8446)

In [22]:
model=MultinomialNB()
model.fit(X_new,y)
pred=model.predict(X_new)
f1_score(y,pred,average=None)

array([0.98857907, 0.98841457])

In [23]:
corpus=df.Msg
target=df.Target

tv=TfidfVectorizer(lowercase=True,stop_words=sws)
smote=SMOTE()
pipe=Pipeline([('vec',tv),('smote',smote),('model',MultinomialNB())])
pipe.fit(corpus,target)

TypeError: All intermediate steps should be transformers and implement fit and transform or be the string 'passthrough' 'SMOTE()' (type <class 'imblearn.over_sampling._smote.base.SMOTE'>) doesn't

In [24]:
from imblearn.pipeline import Pipeline

In [25]:
corpus=df.Msg
target=df.Target

tv=TfidfVectorizer(lowercase=True,stop_words=sws)
smote=SMOTE()
pipe=Pipeline([('vec',tv),('smote',smote),('model',MultinomialNB())])
pipe.fit(corpus,target)

Pipeline(steps=[('vec',
                 TfidfVectorizer(stop_words=['system', 'thin', 'hereafter',
                                             'cant', 'we', 'so', 'your',
                                             'everything', 'whereupon', 'all',
                                             'if', 'such', 'that', 'sometime',
                                             'hundred', 'when', 'four', 'with',
                                             'part', 'therefore', 'being',
                                             'front', 'thus', 'yourself',
                                             'indeed', 'had', 'someone',
                                             'mostly', 'can', 'whenever', ...])),
                ('smote', SMOTE()), ('model', MultinomialNB())])

In [26]:
joblib.dump(pipe,'spam_classifier.pkl')

['spam_classifier.pkl']

In [28]:
df=pd.read_json('c:/ml_datasets/news.json',lines=True)
df

,category,headline,authors,link,short_description,date
0,CRIME,There Were 2 Mass Shootings In Texas Last Week...,Melissa Jeltsen,https://www.huffingtonpost.com/entry/texas-ama...,She left her husband. He killed their children...,2018-05-26
1,ENTERTAINMENT,Will Smith Joins Diplo And Nicky Jam For The 2...,Andy McDonald,https://www.huffingtonpost.com/entry/will-smit...,Of course it has a song.,2018-05-26
2,ENTERTAINMENT,Hugh Grant Marries For The First Time At Age 57,Ron Dicker,https://www.huffingtonpost.com/entry/hugh-gran...,The actor and his longtime girlfriend Anna Ebe...,2018-05-26
3,ENTERTAINMENT,Jim Carrey Blasts 'Castrato' Adam Schiff And D...,Ron Dicker,https://www.huffingtonpost.com/entry/jim-carre...,The actor gives Dems an ass-kicking for not fi...,2018-05-26
4,ENTERTAINMENT,Julianna Margulies Uses Donald Trump Poop Bags...,Ron Dicker,https://www.huffingtonpost.com/entry/julianna-...,"The ""Dietland"" actress said using the bags is ...",2018-05-26
...,...,...,...,...,...,...
1431,BUSINESS,Four More Bank Closures Mark the Week of Janua...,"Dennis Santiago, Contributor\nGlobal Risk and ...",https://www.huffingtonpost.com/entry/four-more...,The general pattern of the FDIC closing banks ...,2012-01-28
1432,BUSINESS,Everything You Need To Know About Overdraft Fe...,Harry Bradford,https://www.huffingtonpost.com/entry/bank-fees...,Don't like keeping all of your money stuffed u...,2012-01-28
1433,BUSINESS,Walmart Waving Goodbye To Some Greeters,,https://www.huffingtonpost.comhttp://jobs.aol....,"After 30 years, ""People Greeters"" will no long...",2012-01-28
1434,BUSINESS,"At World Economic Forum, Fear of Global Contag...","Peter S. Goodman, Contributor\nExecutive Busin...",https://www.huffingtonpost.com/entry/world-eco...,"For decades, as crises have assailed developin...",2012-01-28


In [60]:
df.category.value_counts()

category
POLITICS          516
ENTERTAINMENT     315
COMEDY            101
WORLD NEWS         81
BLACK VOICES       75
QUEER VOICES       71
WEIRD NEWS         41
CRIME              39
MEDIA              37
WOMEN              29
TECH               27
SPORTS             22
IMPACT             19
BUSINESS           16
RELIGION           13
TRAVEL             11
SCIENCE             7
ENVIRONMENT         5
LATINO VOICES       4
EDUCATION           4
CULTURE & ARTS      3
Name: count, dtype: int64

In [29]:
corpus=df.headline
target=df.category

tv=TfidfVectorizer(lowercase=True,stop_words=sws)
smote=SMOTE(k_neighbors=2)
pipe=Pipeline([('vec',tv),('smote',smote),('model',MultinomialNB())])
pipe.fit(corpus,target)

Pipeline(steps=[('vec',
                 TfidfVectorizer(stop_words=['system', 'thin', 'hereafter',
                                             'cant', 'we', 'so', 'your',
                                             'everything', 'whereupon', 'all',
                                             'if', 'such', 'that', 'sometime',
                                             'hundred', 'when', 'four', 'with',
                                             'part', 'therefore', 'being',
                                             'front', 'thus', 'yourself',
                                             'indeed', 'had', 'someone',
                                             'mostly', 'can', 'whenever', ...])),
                ('smote', SMOTE(k_neighbors=2)), ('model', MultinomialNB())])

In [30]:
joblib.dump(pipe,'news_cat.pkl')

['news_cat.pkl']